In [ ]:
import ollama
# HeatUP
MODEL = "llama3.2:3b"

response = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Hello!"}
    ]
)

print(response.message.content)

Hello! How can I assist you today?


In [ ]:
import pydantic_ai_provenance




In [ ]:
"""pydantic-ai-provenance + Ollama demo.

Setup:
    ollama pull qwen3:8b            # or any local model that supports tool calling
    uv pip install pydantic-ai-provenance "pydantic-ai-slim[openai]"   # Python >= 3.12
Run:
    python provenance_ollama_demo.py                 # uses Ollama at localhost:11434
    OLLAMA_MODEL=llama3.1:8b python provenance_ollama_demo.py
    OFFLINE=1 python provenance_ollama_demo.py       # scripted models, tests the wiring only

What it shows:
  1. Provenance graph of the run (source tool -> model -> final output)
  2. Step 1: hallucinated citation keys removed
  3. Step 2: TF-IDF overlap per cited claim (lexical only)
  4. Step 3: LLM entailment judge, wired manually (the library does not call it for you yet)
"""
import asyncio
import os

from pydantic_ai import Agent
from pydantic_ai.messages import ModelResponse, TextPart, ToolCallPart, ToolReturnPart
from pydantic_ai.models.function import AgentInfo, FunctionModel

from pydantic_ai_provenance.attribution import attribute_output
from pydantic_ai_provenance.capability import ProvenanceCapability
from pydantic_ai_provenance.verification import (
    compare_claim_to_sources_tfidf,
    entailment_agent,
    strip_unresolvable_citation_keys,
)

POLICY = (
    "Refund policy. Refunds over 500 EGP require manager approval. "
    "Refunds under 500 EGP are approved by the cashier. "
    "All refunds must be logged in the finance system within 24 hours."
)
QUESTION = "Using the policy, explain who approves refunds. Cite your sources."


def build_models():
    if os.environ.get("OFFLINE"):
        def agent_fn(messages, info: AgentInfo) -> ModelResponse:
            done = any(isinstance(p, ToolReturnPart) for m in messages for p in getattr(m, "parts", []))
            if not done:
                return ModelResponse(parts=[ToolCallPart("read_policy", {"name": "refunds"})])
            return ModelResponse(parts=[TextPart(
                "Refunds over 500 EGP require manager approval. [REF|d_1]\n\n"
                "Refunds under 500 EGP are approved by the manager. [REF|d_1]\n\n"
                "An auditor reviews every refund. [REF|d_9]")])

        def judge_fn(messages, info: AgentInfo) -> ModelResponse:  # fixed score: wiring test only
            return ModelResponse(parts=[ToolCallPart("final_result", {"probability_support": 0.5, "rationale": "offline stub"})])

        return FunctionModel(agent_fn), FunctionModel(judge_fn)

    from pydantic_ai.models.openai import OpenAIChatModel
    from pydantic_ai.providers.ollama import OllamaProvider

    name = os.environ.get("OLLAMA_MODEL", "llama3.2:3b")
    base = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434/v1")
    model = OpenAIChatModel(name, provider=OllamaProvider(base_url=base))
    return model, model


async def main() -> None:
    agent_model, judge_model = build_models()
    prov = ProvenanceCapability(agent_name="policy_bot", source_tools=["read_policy"])
    agent = Agent(agent_model, capabilities=[prov], system_prompt="Answer only from the policy source.")

    @agent.tool_plain
    def read_policy(name: str) -> str:
        """Return the text of a policy document by name."""
        return POLICY

    result = await agent.run(QUESTION)
    store = prov.store
    print("=== model output ===\n" + result.output)
    print("\n=== provenance ===\n" + attribute_output(store).summary())

    sanitized, removed = strip_unresolvable_citation_keys(result.output, store)
    print("\n=== step 1: removed keys ===")
    for r in removed:
        print(f"{r.raw_tag}: removed {r.removed_keys}")

    judge = entailment_agent(judge_model)
    print("\n=== step 2 (TF-IDF) and step 3 (LLM judge) per cited claim ===")
    for rec in compare_claim_to_sources_tfidf(sanitized, store):
        tfidf = max(rec.scores) if rec.scores else 0.0
        premise = rec.source_excerpts[0] if rec.source_excerpts else ""
        verdict = await judge.run(f"Premise (source excerpt):\n{premise}\n\nHypothesis (cited claim):\n{rec.claim_excerpt}")
        j = verdict.output
        print(f"- {rec.claim_excerpt[:60]!r}\n  tfidf={tfidf:.2f}  judge={j.probability_support:.2f}  ({j.rationale})")


asyncio.run(main())